[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/067.%20The%20Kitting%20%26%20Value-Added%20Service%20Scheduling%20Problem/P67-Tier-3_executed.ipynb)

---

# 67. The Kitting and Value-Added Service Scheduling Problem
## Tier 3: Genetic Algorithm

### Purpose and scope

This notebook uses a genetic algorithm (GA) to search for a kitting production schedule. A schedule determines how many units of each kit are produced on each shift in each period. The algorithm evaluates schedules using a weighted fitness score that combines cost, service level, capacity use, and load balance. It does not guarantee a globally optimal schedule.

The example has five kits, three shifts, and seven periods. Its inputs include kit processing times and demand, shift-period capacities and labor costs, and kit-period holding-cost rates. The main outputs are the best schedule found, its objective metrics, and records of fitness and population diversity across generations. A greedy baseline and random-schedule trials provide context for the GA result; they do not establish that the GA is universally superior.

### Modeling assumptions

- Production quantities are nonnegative integers. Production in a kit-period cannot exceed that period's demand.
- Processing time is measured in minutes per unit, and labor cost is interpreted as dollars per labor hour. Thus, labor cost per unit is processing time multiplied by the hourly rate and divided by 60.
- Capacity is measured in available labor minutes for each shift-period. Each shift-period capacity is enforced separately.
- Holding cost is a per-unit proxy applied to production in its period. The model does not track inventory carried between periods, kit-component availability, setup times, or precedence relationships.
- Shortage is measured over the full planning horizon for each kit. The model uses a penalty for unmet demand rather than a separate decision variable for backorders.
- The supplied `cost_savings` values are used to prioritize the greedy baseline. They are not subtracted from the GA's cost objective because the fitness model defines cost as labor, holding, and shortage costs.

### Genetic algorithm in this notebook

A chromosome is a candidate schedule, encoded as a three-dimensional array. The population is the collection of candidate schedules considered at one stage of the search. Each generation creates a new population through tournament selection, uniform crossover, mutation, and constraint repair. Elitism carries a small number of the best candidates forward unchanged.

The algorithm follows this order: create the problem data; initialize a population; evaluate fitness; repeatedly select parents, recombine and mutate their genes, repair feasibility, and evaluate offspring; then analyze and visualize the best result. The fixed random seeds make the stochastic sequence reproducible in a given software environment.

The example retains a population of 50 and runs 500 generations. These are search settings, not evidence of a particular cost reduction or service level. Actual results are produced only when the notebook is run.

## 1. Mathematical model and fitness

### Schedule representation and feasibility

Let $x_{kst}$ be the number of units of kit $k$ produced on shift $s$ in period $t$. A chromosome stores these decisions in `genes[k, s, t]`. The model requires integer, nonnegative production, limits production by period demand, and limits the processing time assigned to each shift-period by its capacity:

$$
x_{kst} \in \mathbb{Z}_{\geq 0}, \qquad
\sum_s x_{kst} \leq d_{kt} \quad \forall k,t,
$$

$$
\sum_k p_k x_{kst} \leq C_{st} \quad \forall s,t.
$$

Here, $d_{kt}$ is demand, $p_k$ is processing time per unit, and $C_{st}$ is available capacity. The repair function enforces these restrictions after crossover and mutation. The initial-population generator also respects the shift-period capacities and never assigns more than the remaining demand.

### Objective metrics

The cost metric is the sum of labor cost, the holding-cost proxy, and the horizon-wide shortage penalty:

$$
\begin{aligned}
L(x) &= \sum_{k,s,t} x_{kst}p_k\frac{w_{st}}{60}, \\
H(x) &= \sum_{k,s,t} x_{kst}h_{kt}, \\
S(x) &= \sum_k \max\left(0,\sum_t d_{kt}-\sum_{s,t}x_{kst}\right), \\
C(x) &= L(x)+H(x)+\lambda S(x).
\end{aligned}
$$

$w_{st}$ is the labor cost per hour, $h_{kt}$ is the holding-cost proxy per produced unit, and $\lambda$ is the shortage penalty per unit. The division by 60 converts processing minutes to hours. In the code, these quantities are calculated in `calculate_multi_objective_fitness`.

Service level is total production divided by total demand, expressed as a percentage. Capacity utilization is total processing minutes used divided by total available minutes, also expressed as a percentage. To discourage schedules above 95% aggregate utilization, the fitness uses a penalized utilization score: above 95%, the excess percentage points are multiplied by 2 and subtracted from the raw utilization percentage. The reported `capacity_utilization` remains the physical utilization; `utilization_score` is the penalized score used by fitness.

Load balance is based on shift-level utilization. If $u_s$ is the percentage utilization of shift $s$, the coefficient of variation is $\mathrm{CV}=\sigma(u)/\mu(u)$ when the mean is positive. The score is $\max(0,100-100\mathrm{CV})$; if all shifts are unused, the implementation assigns a score of 100, following the stated formula's zero-mean convention.

### Weighted fitness

The genetic algorithm maximizes a scalar fitness formed by a weighted sum of normalized metrics:

$$
F(x)=0.4\frac{1}{1+C(x)/1000}
+0.3\frac{\mathrm{Service}(x)}{100}
+0.2\frac{\mathrm{UtilScore}(x)}{100}
+0.1\frac{\mathrm{Balance}(x)}{100}.
$$

The cost term is inverted so that lower cost produces a larger fitness contribution. The other terms are scaled to a 0-to-1 percentage scale. These weights encode the example's priorities; they do not make the four underlying metrics interchangeable. The implementation is `calculate_aggregated_fitness`.

### Search operators and computational effort

Tournament selection samples candidates and chooses the one with the highest fitness. Uniform crossover independently chooses which parent's value supplies each gene. Mutation changes selected production quantities, and repair restores feasibility. Elitism preserves the best candidates. The algorithm is stochastic and uses a fixed weighted fitness, not a Pareto-front method.

If $G$ is the number of generations, $P$ the population size, and $L$ the number of genes per chromosome, evaluating and varying the population takes roughly $O(GPL)$ work. The pairwise population-diversity calculation takes $O(GP^2L)$ work. These expressions describe the implementation's dominant operations; actual runtime also depends on the Python environment.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the stochastic search reproducible in a given environment.
np.random.seed(42)
random.seed(42)

## 2. Define the example problem

The data arrays follow the same index order as the schedule: kit, shift, and period where applicable. Processing times are minutes per unit; capacities are minutes per shift-period; labor rates are dollars per hour; demand is units per kit-period. The GA settings control population size, number of generations, and the selection and variation operators.

In [ ]:
@dataclass
class GAProblem:
    """Inputs and search settings for the genetic algorithm example."""

    num_kits: int
    num_shifts: int
    num_periods: int
    processing_times: np.ndarray
    cost_savings: np.ndarray
    demand: np.ndarray
    capacities: np.ndarray
    labor_costs: np.ndarray
    holding_costs: np.ndarray
    shortage_penalty: float = 10.0
    population_size: int = 50
    max_generations: int = 500
    crossover_rate: float = 0.8
    mutation_rate: float = 0.1
    tournament_size: int = 3
    elitism_ratio: float = 0.1


@dataclass
class Chromosome:
    """A candidate production schedule and its scalar fitness."""

    genes: np.ndarray
    fitness: float = 0.0
    rank: int = 0
    crowding_distance: float = 0.0

    def copy(self) -> "Chromosome":
        """Copy the schedule array so operators cannot alter the source."""
        return Chromosome(
            genes=self.genes.copy(),
            fitness=self.fitness,
            rank=self.rank,
            crowding_distance=self.crowding_distance,
        )


def create_ga_example() -> GAProblem:
    """Create the fixed five-kit, three-shift, seven-period example."""
    return GAProblem(
        num_kits=5,
        num_shifts=3,
        num_periods=7,
        processing_times=np.array([3, 5, 4, 6, 7]),
        cost_savings=np.array([2.50, 4.00, 3.20, 5.50, 4.80]),
        demand=np.array(
            [
                [45, 50, 40, 55, 48, 52, 47],
                [30, 35, 32, 38, 34, 36, 33],
                [25, 28, 26, 30, 27, 29, 25],
                [20, 22, 21, 24, 23, 25, 22],
                [15, 18, 16, 19, 17, 20, 18],
            ]
        ),
        capacities=np.array(
            [
                [480, 480, 480, 480, 480, 480, 480],
                [420, 420, 420, 420, 420, 420, 420],
                [360, 360, 360, 360, 360, 360, 360],
            ]
        ),
        labor_costs=np.array(
            [
                [15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0],
                [18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0],
                [22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0],
            ]
        ),
        holding_costs=np.array(
            [
                [0.80, 0.80, 0.80, 0.80, 0.80, 0.80, 0.80],
                [1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00],
                [0.90, 0.90, 0.90, 0.90, 0.90, 0.90, 0.90],
                [1.20, 1.20, 1.20, 1.20, 1.20, 1.20, 1.20],
                [1.10, 1.10, 1.10, 1.10, 1.10, 1.10, 1.10],
            ]
        ),
        shortage_penalty=10.0,
        population_size=50,
        max_generations=500,
        crossover_rate=0.8,
        mutation_rate=0.1,
        tournament_size=3,
        elitism_ratio=0.1,
    )


problem = create_ga_example()

assert problem.processing_times.shape == (problem.num_kits,)
assert problem.cost_savings.shape == (problem.num_kits,)
assert problem.demand.shape == (problem.num_kits, problem.num_periods)
assert problem.capacities.shape == (problem.num_shifts, problem.num_periods)
assert problem.labor_costs.shape == (problem.num_shifts, problem.num_periods)
assert problem.holding_costs.shape == (problem.num_kits, problem.num_periods)

print(
    f"Problem created: {problem.num_kits} kits, "
    f"{problem.num_shifts} shifts, {problem.num_periods} periods"
)
print(
    f"Population size: {problem.population_size}; "
    f"maximum generations: {problem.max_generations}"
)

## 3. Initialize a population

A population gives the search several candidate schedules rather than committing to one starting point. For each kit and period, the initializer shuffles shifts with enough remaining capacity and assigns a random feasible quantity. Some demand may remain unmet; the shortage penalty lets fitness account for that. Each chromosome has shape `(number of kits, number of shifts, number of periods)`.

In [ ]:
def generate_random_chromosome(problem: GAProblem) -> Chromosome:
    """Generate a random schedule that respects demand and capacity."""
    genes = np.zeros(
        (problem.num_kits, problem.num_shifts, problem.num_periods),
        dtype=int,
    )
    remaining_demand = problem.demand.copy()
    remaining_capacity = problem.capacities.copy()

    for k in range(problem.num_kits):
        for t in range(problem.num_periods):
            available_shifts = [
                s
                for s in range(problem.num_shifts)
                if remaining_capacity[s, t] >= problem.processing_times[k]
            ]
            np.random.shuffle(available_shifts)

            for s in available_shifts:
                if remaining_demand[k, t] <= 0:
                    break

                max_producible = min(
                    remaining_demand[k, t],
                    int(remaining_capacity[s, t] / problem.processing_times[k]),
                )
                if max_producible > 0:
                    quantity = np.random.randint(0, max_producible + 1)
                    genes[k, s, t] = quantity
                    remaining_demand[k, t] -= quantity
                    remaining_capacity[s, t] -= (
                        quantity * problem.processing_times[k]
                    )

    return Chromosome(genes=genes)


def initialize_population(problem: GAProblem) -> List[Chromosome]:
    """Create the initial population of candidate schedules."""
    return [
        generate_random_chromosome(problem)
        for _ in range(problem.population_size)
    ]


population = initialize_population(problem)
sample = population[0]
print(f"Initialized {len(population)} chromosomes.")
print(f"Sample chromosome shape: {sample.genes.shape}")
print(f"Sample chromosome production: {np.sum(sample.genes):.0f} units")

## 4. Evaluate schedules

The evaluator converts a chromosome into interpretable cost and performance metrics. The scalar fitness function then applies the stated weights so tournament selection can compare candidates on one scale. Because the search maximizes fitness, lower cost contributes more through the inverted cost term, while higher service, utilization score, and balance contribute more through their direct normalized terms.

In [ ]:
def calculate_multi_objective_fitness(
    chromosome: Chromosome,
    problem: GAProblem,
) -> Dict[str, float]:
    """Calculate cost, service, utilization, and balance metrics."""
    genes = chromosome.genes

    production_cost = 0.0
    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                production_cost += (
                    genes[k, s, t]
                    * problem.processing_times[k]
                    * problem.labor_costs[s, t]
                    / 60
                )

    inventory_cost = float(
        np.sum(genes * problem.holding_costs[:, np.newaxis, :])
    )
    total_production_per_kit = np.sum(genes, axis=(1, 2))
    total_demand_per_kit = np.sum(problem.demand, axis=1)
    shortages = np.maximum(0, total_demand_per_kit - total_production_per_kit)
    shortage_cost = float(np.sum(shortages) * problem.shortage_penalty)
    total_cost = production_cost + inventory_cost + shortage_cost

    total_demand = float(np.sum(problem.demand))
    total_production = float(np.sum(genes))
    service_level = (
        total_production / total_demand * 100 if total_demand > 0 else 0.0
    )

    used_capacity = float(
        np.sum(genes * problem.processing_times[:, np.newaxis, np.newaxis])
    )
    total_capacity = float(np.sum(problem.capacities))
    capacity_utilization = (
        used_capacity / total_capacity * 100 if total_capacity > 0 else 0.0
    )
    utilization_penalty = max(0.0, capacity_utilization - 95.0) * 2.0
    utilization_score = capacity_utilization - utilization_penalty

    shift_utilizations = []
    for s in range(problem.num_shifts):
        shift_used = float(
            np.sum(genes[:, s, :] * problem.processing_times[:, np.newaxis])
        )
        shift_capacity = float(np.sum(problem.capacities[s, :]))
        shift_utilizations.append(
            shift_used / shift_capacity * 100 if shift_capacity > 0 else 0.0
        )

    if len(shift_utilizations) > 1:
        mean_utilization = float(np.mean(shift_utilizations))
        std_utilization = float(np.std(shift_utilizations))
        coefficient_of_variation = (
            std_utilization / mean_utilization if mean_utilization > 0 else 0.0
        )
        load_balancing = max(0.0, 100.0 - coefficient_of_variation * 100.0)
    else:
        load_balancing = 100.0

    return {
        "total_cost": float(total_cost),
        "service_level": float(service_level),
        "capacity_utilization": float(capacity_utilization),
        "utilization_score": float(utilization_score),
        "load_balancing": float(load_balancing),
        "total_production": total_production,
    }


def calculate_aggregated_fitness(
    objectives: Dict[str, float],
    weights: Optional[Dict[str, float]] = None,
) -> float:
    """Convert the objective metrics into a maximized weighted score."""
    if weights is None:
        weights = {
            "total_cost": 0.4,
            "service_level": 0.3,
            "capacity_utilization": 0.2,
            "load_balancing": 0.1,
        }

    normalized_cost = 1.0 / (1.0 + objectives["total_cost"] / 1000.0)
    normalized_service = objectives["service_level"] / 100.0
    normalized_utilization = objectives["utilization_score"] / 100.0
    normalized_balance = objectives["load_balancing"] / 100.0

    return float(
        weights["total_cost"] * normalized_cost
        + weights["service_level"] * normalized_service
        + weights["capacity_utilization"] * normalized_utilization
        + weights["load_balancing"] * normalized_balance
    )


sample_objectives = calculate_multi_objective_fitness(sample, problem)
sample_fitness = calculate_aggregated_fitness(sample_objectives)
print("Sample chromosome metrics:")
for name, value in sample_objectives.items():
    if name == "total_cost":
        print(f"  {name}: ${value:.2f}")
    elif name == "total_production":
        print(f"  {name}: {value:.0f} units")
    else:
        print(f"  {name}: {value:.1f}%")
print(f"Sample aggregated fitness: {sample_fitness:.4f}")

## 5. Selection, crossover, mutation, and repair

Selection favors higher-fitness schedules without always choosing only the current best. Crossover recombines parent schedules gene by gene, and mutation introduces additional variation. These operations can produce infeasible quantities, so repair is essential: it rounds quantities to nonnegative integers, enforces each kit-period demand limit, and then enforces each shift-period capacity limit. Repair is part of the GA's candidate-generation process, not a separate optimization method.

In [ ]:
def tournament_selection(
    population: List[Chromosome],
    tournament_size: int,
) -> Chromosome:
    """Return a copy of the fittest candidate in a random tournament."""
    if not population:
        raise ValueError("Tournament selection requires a nonempty population.")
    participants = random.sample(population, min(tournament_size, len(population)))
    return max(participants, key=lambda chromosome: chromosome.fitness).copy()


def uniform_crossover(
    parent1: Chromosome,
    parent2: Chromosome,
    crossover_rate: float,
) -> Tuple[Chromosome, Chromosome]:
    """Create two offspring by independently exchanging schedule genes."""
    if parent1.genes.shape != parent2.genes.shape:
        raise ValueError("Parent chromosomes must have matching shapes.")
    if random.random() > crossover_rate:
        return parent1.copy(), parent2.copy()

    choose_parent1 = np.random.random(parent1.genes.shape) < 0.5
    offspring1_genes = np.where(choose_parent1, parent1.genes, parent2.genes)
    offspring2_genes = np.where(choose_parent1, parent2.genes, parent1.genes)
    return Chromosome(offspring1_genes), Chromosome(offspring2_genes)


def adaptive_mutation(
    chromosome: Chromosome,
    problem: GAProblem,
    mutation_rate: float,
) -> Chromosome:
    """Mutate selected quantities with changes scaled to gene values."""
    mutated = chromosome.copy()
    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                if random.random() >= mutation_rate:
                    continue

                current_value = int(mutated.genes[k, s, t])
                if current_value > 0:
                    mutation_type = random.choice(["increase", "decrease", "zero"])
                    if mutation_type == "increase":
                        increase = random.randint(
                            1, max(1, int(current_value * 0.3))
                        )
                        mutated.genes[k, s, t] = current_value + increase
                    elif mutation_type == "decrease":
                        decrease = random.randint(1, max(1, int(current_value * 0.5)))
                        mutated.genes[k, s, t] = max(0, current_value - decrease)
                    else:
                        mutated.genes[k, s, t] = 0
                else:
                    mutated.genes[k, s, t] = random.randint(1, 5)
    return mutated


def repair_chromosome(
    chromosome: Chromosome,
    problem: GAProblem,
) -> Chromosome:
    """Return a nonnegative integer schedule within demand and capacity."""
    expected_shape = (problem.num_kits, problem.num_shifts, problem.num_periods)
    if chromosome.genes.shape != expected_shape:
        raise ValueError(f"Expected chromosome shape {expected_shape}.")

    repaired = chromosome.copy()
    repaired.genes = np.maximum(0, np.rint(repaired.genes)).astype(int)

    # Enforce the demand limit for each kit-period before checking capacities.
    for k in range(problem.num_kits):
        for t in range(problem.num_periods):
            total_production = int(np.sum(repaired.genes[k, :, t]))
            demand_limit = int(problem.demand[k, t])
            if total_production > demand_limit:
                scale = demand_limit / total_production
                repaired.genes[k, :, t] = np.floor(
                    repaired.genes[k, :, t] * scale
                ).astype(int)

    # Scaling and flooring keeps the labor-minute total within each capacity.
    for s in range(problem.num_shifts):
        for t in range(problem.num_periods):
            used = int(
                np.sum(repaired.genes[:, s, t] * problem.processing_times)
            )
            capacity_limit = int(problem.capacities[s, t])
            if used > capacity_limit:
                scale = capacity_limit / used
                repaired.genes[:, s, t] = np.floor(
                    repaired.genes[:, s, t] * scale
                ).astype(int)

    assert np.all(repaired.genes >= 0)
    assert np.all(repaired.genes == np.floor(repaired.genes))
    assert np.all(np.sum(repaired.genes, axis=1) <= problem.demand)
    for s in range(problem.num_shifts):
        for t in range(problem.num_periods):
            used = np.sum(repaired.genes[:, s, t] * problem.processing_times)
            assert used <= problem.capacities[s, t]
    return repaired


selected = tournament_selection(population, problem.tournament_size)
parent1, parent2 = population[0], population[1]
offspring1, offspring2 = uniform_crossover(
    parent1, parent2, problem.crossover_rate
)
mutated = adaptive_mutation(offspring1, problem, problem.mutation_rate)
repaired = repair_chromosome(mutated, problem)
print(f"Selected candidate fitness: {selected.fitness:.4f}")
print(f"Offspring chromosome shape: {offspring1.genes.shape}")
print(f"Repaired schedule production: {np.sum(repaired.genes):.0f} units")

## 6. Run the genetic algorithm

The initial population is evaluated before evolution begins. In each generation, the algorithm preserves the elite candidates, fills the rest of the population with repaired and evaluated offspring, and records the best fitness, average fitness, best-candidate metrics, and mean pairwise gene distance. The diversity measure summarizes how different the schedules are from one another; it is not itself an objective or a guarantee against premature convergence.

In [ ]:
def genetic_algorithm(problem: GAProblem) -> Dict:
    """Run the population-based genetic search and return its records."""
    start_time = time.time()
    population = initialize_population(problem)

    for chromosome in population:
        objectives = calculate_multi_objective_fitness(chromosome, problem)
        chromosome.fitness = calculate_aggregated_fitness(objectives)
    population.sort(key=lambda chromosome: chromosome.fitness, reverse=True)

    evolution_history = {
        "best_fitness": [],
        "avg_fitness": [],
        "best_objectives": [],
        "diversity": [],
    }

    for generation in range(problem.max_generations):
        elite_size = int(problem.population_size * problem.elitism_ratio)
        new_population = [
            chromosome.copy() for chromosome in population[:elite_size]
        ]

        while len(new_population) < problem.population_size:
            parent1 = tournament_selection(population, problem.tournament_size)
            parent2 = tournament_selection(population, problem.tournament_size)
            offspring1, offspring2 = uniform_crossover(
                parent1, parent2, problem.crossover_rate
            )

            for offspring in (offspring1, offspring2):
                offspring = adaptive_mutation(
                    offspring, problem, problem.mutation_rate
                )
                offspring = repair_chromosome(offspring, problem)
                objectives = calculate_multi_objective_fitness(offspring, problem)
                offspring.fitness = calculate_aggregated_fitness(objectives)
                new_population.append(offspring)
                if len(new_population) >= problem.population_size:
                    break

        population = new_population[: problem.population_size]
        population.sort(key=lambda chromosome: chromosome.fitness, reverse=True)

        best_chromosome = population[0]
        average_fitness = float(np.mean([c.fitness for c in population]))
        pairwise_distance = 0.0
        pair_count = 0
        for i in range(len(population)):
            for j in range(i + 1, len(population)):
                pairwise_distance += float(
                    np.sum(np.abs(population[i].genes - population[j].genes))
                )
                pair_count += 1
        diversity = pairwise_distance / pair_count if pair_count else 0.0

        evolution_history["best_fitness"].append(best_chromosome.fitness)
        evolution_history["avg_fitness"].append(average_fitness)
        evolution_history["best_objectives"].append(
            calculate_multi_objective_fitness(best_chromosome, problem)
        )
        evolution_history["diversity"].append(diversity)

        if generation % 50 == 0:
            print(
                f"Generation {generation + 1:4d}: "
                f"best fitness = {best_chromosome.fitness:.4f}, "
                f"average fitness = {average_fitness:.4f}, "
                f"diversity = {diversity:.1f}"
            )

    best_chromosome = population[0]
    return {
        "best_chromosome": best_chromosome,
        "best_objectives": calculate_multi_objective_fitness(
            best_chromosome, problem
        ),
        "evolution_history": evolution_history,
        "computation_time": time.time() - start_time,
        "final_population": population,
    }


print("Starting the genetic algorithm.")
ga_results = genetic_algorithm(problem)
print("Genetic algorithm run complete.")
print(f"Computation time: {ga_results['computation_time']:.2f} seconds")
print(f"Best fitness: {ga_results['best_chromosome'].fitness:.4f}")
print("Best-candidate metrics:")
for name, value in ga_results["best_objectives"].items():
    if name == "total_cost":
        print(f"  {name}: ${value:.2f}")
    elif name == "total_production":
        print(f"  {name}: {value:.0f} units")
    else:
        print(f"  {name}: {value:.1f}%")

## 7. Inspect the best schedule and search history

The best chromosome is the highest-fitness candidate in the final population, not a proof of global optimality. The schedule listing reports production by kit, shift, and period; the performance summary separates physical utilization from its penalized fitness score. The convergence indicator is an operational diagnostic: it reports the first recorded generation whose best fitness improved by less than 1% over the preceding 50 recorded generations. It does not certify optimality.

In [ ]:
def analyze_ga_results(ga_results: Dict, problem: GAProblem) -> None:
    """Print the schedule, performance metrics, and search diagnostics."""
    best_chromosome = ga_results["best_chromosome"]
    objectives = ga_results["best_objectives"]
    history = ga_results["evolution_history"]

    print("BEST PRODUCTION SCHEDULE")
    print("=" * 70)
    for k in range(problem.num_kits):
        print(
            f"\nKit {k + 1} "
            f"(processing time: {problem.processing_times[k]} min/unit):"
        )
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                quantity = int(best_chromosome.genes[k, s, t])
                if quantity > 0:
                    print(f"  Shift {s + 1}, period {t + 1}: {quantity} units")

        kit_production = int(np.sum(best_chromosome.genes[k, :, :]))
        kit_demand = int(np.sum(problem.demand[k, :]))
        fulfillment = kit_production / kit_demand * 100 if kit_demand else 0.0
        print(
            f"  Kit total: {kit_production}/{kit_demand} units "
            f"({fulfillment:.1f}% fulfilled)"
        )

    print("\nPERFORMANCE METRICS")
    print("=" * 70)
    print(f"Total production: {objectives['total_production']:.0f} units")
    print(f"Total demand: {np.sum(problem.demand):.0f} units")
    print(f"Service level: {objectives['service_level']:.1f}%")
    print(f"Physical capacity utilization: {objectives['capacity_utilization']:.1f}%")
    print(f"Penalized utilization score: {objectives['utilization_score']:.1f}%")
    print(f"Load-balancing score: {objectives['load_balancing']:.1f}%")
    print(f"Total modeled cost: ${objectives['total_cost']:.2f}")
    print(f"Computation time: {ga_results['computation_time']:.2f} seconds")

    best_history = history["best_fitness"]
    average_history = history["avg_fitness"]
    if best_history:
        initial_best = best_history[0]
        final_best = best_history[-1]
        best_improvement = (
            (final_best - initial_best) / abs(initial_best) * 100
            if initial_best != 0
            else 0.0
        )
        initial_average = average_history[0]
        final_average = average_history[-1]
        average_improvement = (
            (final_average - initial_average) / abs(initial_average) * 100
            if initial_average != 0
            else 0.0
        )
        print("\nSEARCH HISTORY")
        print("=" * 70)
        print(f"First recorded best fitness: {initial_best:.4f}")
        print(f"Final best fitness: {final_best:.4f}")
        print(f"Best-fitness change: {best_improvement:.1f}%")
        print(f"Average-fitness change: {average_improvement:.1f}%")

        convergence_generation = None
        for index in range(50, len(best_history)):
            comparison = best_history[index - 50]
            recent_improvement = (
                (best_history[index] - comparison) / abs(comparison) * 100
                if comparison != 0
                else float("inf")
            )
            if recent_improvement < 1.0:
                convergence_generation = index + 1
                break
        if convergence_generation is None:
            print("Convergence diagnostic: threshold not reached in the recorded history.")
        else:
            print(
                "Convergence diagnostic: less than 1% best-fitness improvement "
                f"over 50 recorded generations by generation {convergence_generation}."
            )

    print("\nCAPACITY USE BY SHIFT AND PERIOD")
    print("=" * 70)
    for s in range(problem.num_shifts):
        print(f"\nShift {s + 1}:")
        for t in range(problem.num_periods):
            used = float(
                np.sum(
                    best_chromosome.genes[:, s, t] * problem.processing_times
                )
            )
            capacity = float(problem.capacities[s, t])
            utilization = used / capacity * 100 if capacity > 0 else 0.0
            print(
                f"  Period {t + 1}: {used:.1f}/{capacity:.1f} minutes "
                f"({utilization:.1f}%)"
            )

    # Independently verify that the returned schedule satisfies the model limits.
    assert np.all(best_chromosome.genes >= 0)
    assert np.all(np.sum(best_chromosome.genes, axis=1) <= problem.demand)
    for s in range(problem.num_shifts):
        for t in range(problem.num_periods):
            used = np.sum(
                best_chromosome.genes[:, s, t] * problem.processing_times
            )
            assert used <= problem.capacities[s, t]


analyze_ga_results(ga_results, problem)

## 8. Visualize the search and schedule

The plots show best and average fitness, population diversity, selected objective trajectories, and total production by kit and shift. Cost is scaled to the observed range only for visual comparison; this plotting transformation does not change the fitness calculation. If all recorded costs are equal, the plotted normalized cost is defined as zero rather than dividing by zero.

In [ ]:
def create_ga_visualization(ga_results: Dict, problem: GAProblem):
    """Plot fitness, diversity, objective history, and aggregate production."""
    history = ga_results["evolution_history"]
    best_chromosome = ga_results["best_chromosome"]
    generations = np.arange(1, len(history["best_fitness"]) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Genetic Algorithm: Kitting Schedule Results", fontsize=16)

    ax1 = axes[0, 0]
    ax1.plot(generations, history["best_fitness"], label="Best fitness")
    ax1.plot(generations, history["avg_fitness"], "--", label="Average fitness")
    ax1.set_title("Fitness by generation")
    ax1.set_xlabel("Generation")
    ax1.set_ylabel("Fitness (higher is better)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    convergence_generation = None
    best_history = history["best_fitness"]
    for index in range(50, len(best_history)):
        comparison = best_history[index - 50]
        recent_improvement = (
            (best_history[index] - comparison) / abs(comparison) * 100
            if comparison != 0
            else float("inf")
        )
        if recent_improvement < 1.0:
            convergence_generation = index + 1
            break
    if convergence_generation is not None:
        ax1.axvline(
            convergence_generation,
            color="green",
            linestyle=":",
            label="Convergence diagnostic",
        )
        ax1.legend()

    ax2 = axes[0, 1]
    ax2.plot(generations, history["diversity"], color="green")
    ax2.set_title("Population diversity by generation")
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Mean pairwise gene distance")
    ax2.grid(True, alpha=0.3)
    if len(history["diversity"]) > 100:
        initial_diversity = float(np.mean(history["diversity"][:10]))
        final_diversity = float(np.mean(history["diversity"][-10:]))
        diversity_change = (
            (initial_diversity - final_diversity) / initial_diversity * 100
            if initial_diversity != 0
            else 0.0
        )
        ax2.text(
            0.05,
            0.95,
            f"Diversity decrease: {diversity_change:.1f}%",
            transform=ax2.transAxes,
            va="top",
            bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.8},
        )

    ax3 = axes[1, 0]
    cost_history = np.array(
        [obj["total_cost"] for obj in history["best_objectives"]], dtype=float
    )
    service_history = np.array(
        [obj["service_level"] for obj in history["best_objectives"]], dtype=float
    )
    utilization_history = np.array(
        [obj["capacity_utilization"] for obj in history["best_objectives"]],
        dtype=float,
    )
    cost_range = float(np.max(cost_history) - np.min(cost_history))
    cost_normalized = (
        (cost_history - np.min(cost_history)) / cost_range
        if cost_range > 0
        else np.zeros_like(cost_history)
    )
    ax3.plot(generations, cost_normalized, label="Cost (observed-range scaled)")
    ax3.plot(generations, service_history / 100, label="Service level")
    ax3.plot(generations, utilization_history / 100, label="Physical utilization")
    ax3.set_title("Best-candidate objective history")
    ax3.set_xlabel("Generation")
    ax3.set_ylabel("Scaled value")
    ax3.set_ylim(0, 1.05)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    production_matrix = np.sum(best_chromosome.genes, axis=2)
    sns.heatmap(
        production_matrix,
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        xticklabels=[f"Shift {s + 1}" for s in range(problem.num_shifts)],
        yticklabels=[f"Kit {k + 1}" for k in range(problem.num_kits)],
        cbar_kws={"label": "Total production (units)"},
        ax=axes[1, 1],
    )
    axes[1, 1].set_title("Total production by kit and shift")
    axes[1, 1].set_xlabel("Shift")
    axes[1, 1].set_ylabel("Kit")

    fig.tight_layout()
    plt.show()
    return fig


fig = create_ga_visualization(ga_results, problem)

## 9. Compare with simple baselines

The greedy baseline orders kits by the supplied savings-per-processing-minute ratio, then assigns each kit-period's demand to shifts with available capacity. The random baseline uses the same feasible random-schedule generator as population initialization. Ten random trials are summarized by their mean metrics. The comparisons use the same evaluator and weighted fitness as the GA; they are specific to this data, these settings, and these random trials.

In [ ]:
def compare_with_baselines(problem: GAProblem, ga_results: Dict) -> Dict:
    """Compare the GA with a greedy schedule and random feasible schedules."""
    priorities = problem.cost_savings / problem.processing_times
    kit_order = np.argsort(priorities)[::-1]
        
    genes = np.zeros(
        (problem.num_kits, problem.num_shifts, problem.num_periods), dtype=int
    )
    remaining_capacity = problem.capacities.copy()

    for k in kit_order:
        for t in range(problem.num_periods):
            remaining_demand = int(problem.demand[k, t])
            while remaining_demand > 0:
                available_shifts = [
                    s
                    for s in range(problem.num_shifts)
                    if remaining_capacity[s, t] >= problem.processing_times[k]
                ]
                if not available_shifts:
                    break
                best_shift = max(
                    available_shifts,
                    key=lambda s: remaining_capacity[s, t],
                )
                quantity = min(
                    remaining_demand,
                    int(remaining_capacity[best_shift, t] / problem.processing_times[k]),
                )
                genes[k, best_shift, t] += quantity
                remaining_demand -= quantity
                remaining_capacity[best_shift, t] -= (
                    quantity * problem.processing_times[k]
                )

    greedy_solution = Chromosome(genes=genes)
    greedy_objectives = calculate_multi_objective_fitness(greedy_solution, problem)
    greedy_fitness = calculate_aggregated_fitness(greedy_objectives)

    random_objectives_list = []
    random_fitnesses = []
    for _ in range(10):
        random_solution = generate_random_chromosome(problem)
        objectives = calculate_multi_objective_fitness(random_solution, problem)
        random_objectives_list.append(objectives)
        random_fitnesses.append(calculate_aggregated_fitness(objectives))

    average_random_objectives = {
        key: float(np.mean([obj[key] for obj in random_objectives_list]))
        for key in random_objectives_list[0]
    }
    average_random_fitness = float(np.mean(random_fitnesses))
    ga_objectives = ga_results["best_objectives"]
    ga_fitness = ga_results["best_chromosome"].fitness

    print("BASELINE COMPARISON")
    print("=" * 78)
    print(
        f"{'Method':<16} {'Fitness':>10} {'Cost ($)':>12} "
        f"{'Service (%)':>13} {'Utilization (%)':>17}"
    )
    print("-" * 78)
    rows = [
        ("Random mean", average_random_fitness, average_random_objectives),
        ("Greedy", greedy_fitness, greedy_objectives),
        ("Genetic algorithm", ga_fitness, ga_objectives),
    ]
    for name, fitness, objectives in rows:
        print(
            f"{name:<16} {fitness:>10.4f} "
            f"{objectives['total_cost']:>12.2f} "
            f"{objectives['service_level']:>13.1f} "
            f"{objectives['capacity_utilization']:>17.1f}"
        )

    greedy_cost = greedy_objectives["total_cost"]
    random_cost = average_random_objectives["total_cost"]
    cost_reduction_vs_greedy = (
        (greedy_cost - ga_objectives["total_cost"]) / greedy_cost * 100
        if greedy_cost != 0
        else 0.0
    )
    cost_reduction_vs_random = (
        (random_cost - ga_objectives["total_cost"]) / random_cost * 100
        if random_cost != 0
        else 0.0
    )
    print("\nGA comparison with the baselines:")
    print(f"  Cost change versus greedy: {cost_reduction_vs_greedy:.1f}% reduction")
    print(f"  Service-level difference versus greedy: "
          f"{ga_objectives['service_level'] - greedy_objectives['service_level']:.1f} percentage points")
    print(f"  Cost change versus random mean: {cost_reduction_vs_random:.1f}% reduction")
    print(f"  Service-level difference versus random mean: "
          f"{ga_objectives['service_level'] - average_random_objectives['service_level']:.1f} percentage points")

    methods = ["Random mean", "Greedy", "Genetic algorithm"]
    costs = [
        average_random_objectives["total_cost"],
        greedy_objectives["total_cost"],
        ga_objectives["total_cost"],
    ]
    services = [
        average_random_objectives["service_level"],
        greedy_objectives["service_level"],
        ga_objectives["service_level"],
    ]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    bars1 = ax1.bar(methods, costs, color=["lightcoral", "orange", "green"])
    ax1.set_title("Modeled total cost by method")
    ax1.set_ylabel("Total cost ($)")
    ax1.tick_params(axis="x", rotation=15)
    for bar, cost in zip(bars1, costs):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${cost:.0f}",
            ha="center",
            va="bottom",
        )

    bars2 = ax2.bar(methods, services, color=["lightcoral", "orange", "green"])
    ax2.set_title("Service level by method")
    ax2.set_ylabel("Service level (%)")
    ax2.set_ylim(0, 100)
    ax2.tick_params(axis="x", rotation=15)
    for bar, service in zip(bars2, services):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{service:.1f}%",
            ha="center",
            va="bottom",
        )
    fig.tight_layout()
    plt.show()

    return {
        "ga_objectives": ga_objectives,
        "greedy_objectives": greedy_objectives,
        "random_objectives": average_random_objectives,
    }


comparison_results = compare_with_baselines(problem, ga_results)

## 10. Interpretation and limitations

The final schedule is the best candidate the configured GA found under the weighted fitness function. A larger fitness does not necessarily mean that every individual metric is better: the weights allow trade-offs among modeled cost, service, utilization score, and load balance. Compare the metric values as well as the aggregate score, and interpret the baseline results only for this instance and these settings.

A GA uses stochastic search and provides no optimality certificate. Its outcome can depend on the seed, population size, generation limit, operator rates, repair behavior, and fitness weights. A flat best-fitness history is a search diagnostic rather than proof that the global optimum has been reached. The reported random seeds support reproducibility, but exact behavior can still depend on library versions and execution environment.

The model is intentionally limited to the stated production, demand, labor, holding-cost proxy, and capacity structure. It does not represent inventory flows between periods, kit-component availability, setup decisions, or other operational rules. Those assumptions should be assessed before using a schedule for real operations. Additional realistic constraints or alternative input scenarios can be studied within the same genetic-algorithm framework; this example does not implement them.